# Day 4 – Data Integration & Schema Design

In this notebook, I inspect, clean, and prepare the SAT results dataset for integration into the PostgreSQL database.

Steps covered:
- dataset inspection
- data cleaning
- validation of score ranges
- schema design
- creation of a target table
- insertion of cleaned records into the database

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import psycopg2

In [2]:
df = pd.read_csv("sat-results.csv")
df.head()

,DBN,SCHOOL NAME,Num of SAT Test Takers,SAT Critical Reading Avg. Score,SAT Math Avg. Score,SAT Writing Avg. Score,SAT Critical Readng Avg. Score,internal_school_id,contact_extension,pct_students_tested,academic_tier_rating
0,01M292,HENRY STREET SCHOOL FOR INTERNATIONAL STUDIES,29,355,404,363,355,218160,x345,78%,2.0
1,01M448,UNIVERSITY NEIGHBORHOOD HIGH SCHOOL,91,383,423,366,383,268547,x234,NaN,3.0
2,01M450,EAST SIDE COMMUNITY SCHOOL,70,377,402,370,377,236446,x123,NaN,3.0
3,01M458,FORSYTH SATELLITE ACADEMY,7,414,401,359,414,427826,x123,92%,4.0
4,01M509,MARTA VALLE HIGH SCHOOL,44,390,433,384,390,672714,x123,92%,2.0


In [3]:
df.shape

(493, 11)

In [4]:
df.columns

Index(['DBN', 'SCHOOL NAME', 'Num of SAT Test Takers',
       'SAT Critical Reading Avg. Score', 'SAT Math Avg. Score',
       'SAT Writing Avg. Score', 'SAT Critical Readng Avg. Score',
       'internal_school_id', 'contact_extension', 'pct_students_tested',
       'academic_tier_rating'],
      dtype='object')

In [5]:
print(df.columns.tolist())

['DBN', 'SCHOOL NAME', 'Num of SAT Test Takers', 'SAT Critical Reading Avg. Score', 'SAT Math Avg. Score', 'SAT Writing Avg. Score', 'SAT Critical Readng Avg. Score', 'internal_school_id', 'contact_extension', 'pct_students_tested', 'academic_tier_rating']


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 493 entries, 0 to 492
Data columns (total 11 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   DBN                              493 non-null    object 
 1   SCHOOL NAME                      493 non-null    object 
 2   Num of SAT Test Takers           493 non-null    object 
 3   SAT Critical Reading Avg. Score  493 non-null    object 
 4   SAT Math Avg. Score              493 non-null    object 
 5   SAT Writing Avg. Score           493 non-null    object 
 6   SAT Critical Readng Avg. Score   493 non-null    object 
 7   internal_school_id               493 non-null    int64  
 8   contact_extension                388 non-null    object 
 9   pct_students_tested              376 non-null    object 
 10  academic_tier_rating             402 non-null    float64
dtypes: float64(1), int64(1), object(9)
memory usage: 42.5+ KB


In [7]:
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace(".", "", regex=False)
)

print(df.columns.tolist())

['dbn', 'school_name', 'num_of_sat_test_takers', 'sat_critical_reading_avg_score', 'sat_math_avg_score', 'sat_writing_avg_score', 'sat_critical_readng_avg_score', 'internal_school_id', 'contact_extension', 'pct_students_tested', 'academic_tier_rating']


In [8]:
df = df.drop(columns=[
    "sat_critical_readng_avg_score",
    "internal_school_id",
    "contact_extension"
], errors="ignore")

In [9]:
df = df.replace("s", np.nan)
df = df.replace("N/A", np.nan)

In [10]:
numeric_cols = [
    "num_of_sat_test_takers",
    "sat_critical_reading_avg_score",
    "sat_math_avg_score",
    "sat_writing_avg_score",
    "academic_tier_rating"
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

In [11]:
df["pct_students_tested"] = (
    df["pct_students_tested"]
    .astype(str)
    .str.replace("%", "", regex=False)
    .replace("nan", np.nan)
)

df["pct_students_tested"] = pd.to_numeric(df["pct_students_tested"], errors="coerce")

In [12]:
df = df[
    df["sat_critical_reading_avg_score"].between(200, 800) &
    df["sat_math_avg_score"].between(200, 800) &
    df["sat_writing_avg_score"].between(200, 800)
]

In [13]:
df = df.drop_duplicates(subset="dbn")

In [14]:
score_cols = [
    "sat_critical_reading_avg_score",
    "sat_math_avg_score",
    "sat_writing_avg_score"
]

for col in score_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

In [15]:
df = df[
    (df["sat_critical_reading_avg_score"].between(200,800)) &
    (df["sat_math_avg_score"].between(200,800)) &
    (df["sat_writing_avg_score"].between(200,800))
]

In [16]:
columns_keep = [
    "dbn",
    "school_name",
    "num_of_sat_test_takers",
    "sat_critical_reading_avg_score",
    "sat_math_avg_score",
    "sat_writing_avg_score",
    "pct_students_tested",
    "academic_tier_rating"
]

df = df[columns_keep]
df.head()

,dbn,school_name,num_of_sat_test_takers,sat_critical_reading_avg_score,sat_math_avg_score,sat_writing_avg_score,pct_students_tested,academic_tier_rating
0,01M292,HENRY STREET SCHOOL FOR INTERNATIONAL STUDIES,29.0,355.0,404.0,363.0,78.0,2.0
1,01M448,UNIVERSITY NEIGHBORHOOD HIGH SCHOOL,91.0,383.0,423.0,366.0,NaN,3.0
2,01M450,EAST SIDE COMMUNITY SCHOOL,70.0,377.0,402.0,370.0,NaN,3.0
3,01M458,FORSYTH SATELLITE ACADEMY,7.0,414.0,401.0,359.0,92.0,4.0
4,01M509,MARTA VALLE HIGH SCHOOL,44.0,390.0,433.0,384.0,92.0,2.0


In [17]:
df["num_of_sat_test_takers"] = pd.to_numeric(
    df["num_of_sat_test_takers"],
    errors="coerce"
)

In [18]:
df = df[
    (df["sat_critical_reading_avg_score"].between(200,800)) &
    (df["sat_math_avg_score"].between(200,800)) &
    (df["sat_writing_avg_score"].between(200,800))
]

In [19]:
output_path = Path("cleaned_sat_results.csv")
df.to_csv(output_path, index=False)
print(f"Saved cleaned file to: {output_path.resolve()}")

Saved cleaned file to: /content/cleaned_sat_results.csv


## Schema Design

The SAT dataset can be linked to existing tables using the `dbn` column, which is a unique identifier for NYC schools.

Proposed table:

nyc_schools.julia_sat_scores

Columns:
- dbn
- school_name
- num_of_sat_test_takers
- sat_critical_reading_avg_score
- sat_math_avg_score
- sat_writing_avg_score

In [23]:
import psycopg2

conn = psycopg2.connect(
    dbname="neondb",
    user="neondb_owner",
    password="my_password",
    host="ep-falling-glitter-a5m0j5gk-pooler.us-east-2.aws.neon.tech",
    port="5432",
    sslmode="require"
)

cur = conn.cursor()

print("Connection established")

Connection established


## Database Insertion Script

The following script demonstrates how the cleaned SAT dataset would be inserted into the PostgreSQL database.

The connection parameters are placeholders to avoid exposing credentials.  
In a real environment, valid database credentials would be used.

In [24]:
create_table_query = """
CREATE TABLE IF NOT EXISTS nyc_schools.julia_sat_scores (
    dbn TEXT PRIMARY KEY,
    school_name TEXT,
    num_of_sat_test_takers INTEGER,
    sat_critical_reading_avg_score INTEGER,
    sat_math_avg_score INTEGER,
    sat_writing_avg_score INTEGER,
    pct_students_tested NUMERIC,
    academic_tier_rating NUMERIC
);
"""

cur.execute(create_table_query)
conn.commit()

print("Table ensured")

Table ensured


In [25]:
insert_query = """
INSERT INTO nyc_schools.julia_sat_scores (
    dbn,
    school_name,
    num_of_sat_test_takers,
    sat_critical_reading_avg_score,
    sat_math_avg_score,
    sat_writing_avg_score,
    pct_students_tested,
    academic_tier_rating
)
VALUES (%s,%s,%s,%s,%s,%s,%s,%s)
ON CONFLICT (dbn) DO NOTHING;
"""

In [26]:
for _, row in df.iterrows():
    cur.execute(insert_query, tuple(row))

conn.commit()

print(f"{len(df)} rows inserted")

416 rows inserted


In [27]:
cur.execute("SELECT COUNT(*) FROM nyc_schools.julia_sat_scores;")
print(cur.fetchone())

(416,)


In [28]:
cur.close()
conn.close()

## Data Cleaning Summary

Cleaning steps:

- normalized column names
- converted SAT score columns to numeric values
- removed invalid SAT scores outside the 200–800 range
- handled missing values using `coerce`
- removed duplicate schools using `dbn`

The cleaned dataset is exported as `cleaned_sat_results.csv` and prepared for database insertion.